# Bếp Finance — Python, SQL, and the data pipeline in Colab
Upload the project source as a ZIP file and start with the sample data. Download your results before ending the Colab session. The notebook is intended for analysis and does not run a production service.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','pandas>=2.2,<4','numpy>=1.26,<3','openpyxl>=3.1,<4','pypdf>=5,<7'])


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded=files.upload()  # Select the project ZIP, not an invoice at this step.
archive=next(name for name in uploaded if name.endswith('.zip'))
target=Path('/content/bep_project');target.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
 for member in z.infolist():
  dest=(target/member.filename).resolve()
  if not dest.is_relative_to(target.resolve()): raise ValueError('Unsafe archive path')
 z.extractall(target)
project=target/'bep-finance'
os.chdir(project)
sys.path.insert(0,str(project))


In [ ]:
from backend.pipeline.runner import run
from backend.analytics import connect, monthly_report, forecast
import pandas as pd, json, hashlib, secrets
from backend.data_quality import review_period
os.environ.setdefault('BEP_AUDIT_KEY',secrets.token_urlsafe(48))
lake=Path('/content/bep_lakehouse')
result=run('backend/data/sample_entries.csv',lake,'demo_2024_2025')
print(result)
with connect(lake/'state.sqlite3') as conn:
 display(pd.DataFrame(monthly_report(conn)))
 expected='bronze:'+hashlib.sha256(Path('backend/data/sample_entries.csv').read_bytes()).hexdigest()
 sources={row[0] for row in conn.execute('SELECT DISTINCT source FROM entries')}
 if sources!={expected}:raise ValueError('Mixed sources: review actual data separately; no automatic approval')
 for row in monthly_report(conn,'q1'):review_period(conn,'q1',row['month'],'complete','Bundled synthetic sample only')
 print(json.dumps(forecast(conn,'q1',3,'2026-01-01'),ensure_ascii=False,indent=2))
# Forecast is retrospective Jan-Mar 2026 because sample history ends Dec 2025.


In [ ]:
# Optional: upload a CSV/XLSX/PDF. Excel must have worksheet 'entries'.
# Keep this empty to skip; set a new logical source id when importing a new report.
import_my_file = False
if import_my_file:
 uploaded=files.upload()
 filename=next(iter(uploaded))
 print(run(filename,lake,source_id='my_report_001'))
# PDF returns review_required, not Gold. For reviewed CSV from PDF use same source_id,
# evidence_hash from PDF result and reviewed_by. See pipeline README for revisions.


In [ ]:
# Plot actual SQL/Python totals, not the JavaScript sample dashboard.
with connect(lake/'state.sqlite3') as conn:
 frame=pd.DataFrame(monthly_report(conn,'q1'))
frame.set_index('month')[['revenue_vnd','operating_profit_vnd']].div(1_000_000).plot(figsize=(12,4),title='Q1: revenue / operating profit — synthetic sample',ylabel='Million VND')


In [ ]:
import shutil
archive=shutil.make_archive('/content/bep_pipeline_results','zip',lake)
files.download(archive)
